In [1]:
import pandas as pd
import lxml
from pathlib import Path
from glob import glob

In [2]:
bases = ["CAPP", "CASS", "CONSTIT", "INCA", "JADE"]
root = "/mnt/data/Murphy/src"

paths = {
    base: glob(f"{root}/{base}/**/*.*", recursive=True)
    for base in bases
}

for base, files in paths.items():
    print(f"{base}: {len(files)} fichiers")

CAPP: 1 fichiers
CASS: 92 fichiers
CONSTIT: 2 fichiers
INCA: 1 fichiers
JADE: 256 fichiers


In [3]:
import math
from glob import glob
from collections import defaultdict
import xml.etree.ElementTree as ET


def strip_ns(tag):
    return tag.rsplit('}', 1)[-1] if '}' in tag else tag


# ── Balises TRANSPARENTES (logique produit, pas métier) ──────────────────────
# Les balises de mise en forme HTML n'ont pas à figurer dans le schéma : elles
# sont "invisibles". Leur texte remonte dans la balise métier parente, et leurs
# propres enfants (métier) remontent aussi comme enfants du parent.
#   nom (minuscule) -> séparateur inséré dans le texte concaténé.
#   <p> sépare par un double saut (paragraphe), <br/> par un simple saut.
TRANSPARENT_TAGS = {"p": "\n\n", "br": "\n"}

# Attributs rencontrés sur une balise transparente : on les IGNORE (la balise est
# invisible) mais on les signale, au cas où une donnée métier s'y cacherait.
_transparent_attr_warnings = set()


def _is_transparent(tag, strip):
    return (strip_ns(tag) if strip else tag) in TRANSPARENT_TAGS


def _warn_transparent_attrs(elem, tag):
    for a in elem.attrib:
        sig = (tag, strip_ns(a))
        if sig not in _transparent_attr_warnings:
            _transparent_attr_warnings.add(sig)
            print(f"⚠ balise transparente <{tag}> porte un attribut '{strip_ns(a)}' "
                  f"— ignoré (donnée métier potentiellement perdue)")


def _has_text_effective(elem, strip):
    """Texte présent pour `elem` en tenant compte des transparents : son propre
    texte, le tail de ses enfants (contenu mixte), et récursivement le texte des
    enfants transparents (dont le contenu remonte dans `elem`)."""
    if (elem.text or "").strip():
        return True
    for c in elem:
        if (c.tail or "").strip():
            return True
        if _is_transparent(c.tag, strip) and _has_text_effective(c, strip):
            return True
    return False


def _effective_children(elem, strip):
    """Enfants EFFECTIFS de `elem` : les enfants transparents sont remplacés par
    leurs propres enfants effectifs (récursivement), les autres passent tels quels."""
    for c in elem:
        if _is_transparent(c.tag, strip):
            _warn_transparent_attrs(c, strip_ns(c.tag) if strip else c.tag)
            yield from _effective_children(c, strip)
        else:
            yield c


class Node:
    __slots__ = ("name", "occ", "sumsq", "min_k", "max_k", "n_present",
                 "children", "attribs", "text_present")

    def __init__(self, name):
        self.name = name
        self.occ = 0          # total d'instances de l'élément (= somme des k)
        self.sumsq = 0        # somme des k^2  (pour std)
        self.min_k = None     # min de k sur les instances de parent où présent
        self.max_k = 0
        self.n_present = 0    # nb d'instances de parent contenant >=1 de cet enfant
        self.children = {}
        self.attribs = {}     # clé d'attribut -> Node (présence 0/1 par instance)
        self.text_present = 0 # nb d'instances de ce nœud portant du texte (text ou tail)

    def child(self, name):
        c = self.children.get(name)
        if c is None:
            c = Node(name)
            self.children[name] = c
        return c

    def attrib_child(self, name):
        c = self.attribs.get(name)
        if c is None:
            c = Node(name)
            self.attribs[name] = c
        return c

    def observe(self, k):
        # appelé une fois par instance de parent ; k = nb d'enfants de ce nom (k>=1)
        self.occ += k
        self.sumsq += k * k
        self.n_present += 1
        if k > self.max_k:
            self.max_k = k
        if self.min_k is None or k < self.min_k:
            self.min_k = k


def _merge(elem, node, strip):
    # attributs : une clé apparaît au plus une fois par élément -> observe(1)
    for a in elem.attrib:
        an = strip_ns(a) if strip else a
        node.attrib_child(an).observe(1)

    # texte : propre texte, tail des enfants (mixte), ET texte des transparents
    # dont le contenu remonte ici (p/br invisibles).
    if _has_text_effective(elem, strip):
        node.text_present += 1

    # enfants EFFECTIFS : les transparents sont dépliés dans le parent, donc
    # jamais matérialisés en Node. Regroupés par tag pour compter k par instance.
    groups = defaultdict(list)
    for c in _effective_children(elem, strip):
        t = strip_ns(c.tag) if strip else c.tag
        groups[t].append(c)
    for name, lst in groups.items():
        cn = node.child(name)
        cn.observe(len(lst))
        for c in lst:
            _merge(c, cn, strip)


def _doc_entries(root, split_level, strip):
    # -> dict: key_tuple -> liste des éléments d'entrée pour ce document
    if split_level == 0:
        return {(): [root]}
    res = defaultdict(list)

    def rec(elem, path):
        t = strip_ns(elem.tag) if strip else elem.tag
        newpath = path + (t,)
        if len(newpath) == split_level:
            res[newpath].append(elem)
        else:
            for c in elem:
                rec(c, newpath)

    rec(root, ())
    return res


def aggregate_base(paths, split_level=0, strip_namespaces=True):
    forest = {}          # key tuple -> Node racine
    ndocs = 0
    for p in paths:
        try:
            root = ET.parse(p).getroot()
        except ET.ParseError:
            continue
        ndocs += 1

        if split_level == 0:
            # la racine XML EST le nœud racine de l'arbre ; clé = son nom
            rtag = strip_ns(root.tag) if strip_namespaces else root.tag
            key = (rtag,)
            rnode = forest.get(key)
            if rnode is None:
                rnode = Node(rtag)
                forest[key] = rnode
            rnode.observe(1)               # une instance de racine par document
            _merge(root, rnode, strip_namespaces)
        else:
            for key, entries in _doc_entries(root, split_level, strip_namespaces).items():
                rnode = forest.get(key)
                if rnode is None:
                    rnode = Node(key[-1])
                    forest[key] = rnode
                rnode.observe(len(entries))
                for e in entries:
                    _merge(e, rnode, strip_namespaces)
    return forest, ndocs


def _text_status(node):
    # statut texte dérivé de text_present / occ
    if node.text_present == 0:
        status = "never"
    elif node.text_present == node.occ:
        status = "always"
    else:
        status = "sometimes"
    return {
        "status": status,
        "present": node.text_present,
        "total": node.occ,
        "ratio": node.text_present / node.occ if node.occ else 0.0,
    }


def _serialize(node, n_parent):
    # n_parent = nb d'instances de parent (zéros inclus)
    avg = node.occ / n_parent if n_parent else 0.0
    var = max(node.sumsq / n_parent - avg * avg, 0.0) if n_parent else 0.0
    mn = node.min_k if node.n_present == n_parent else 0   # absent qq part -> min 0
    return {
        "name": node.name,
        "occ": node.occ,
        "min": mn if mn is not None else 0,
        "max": node.max_k,
        "avg": avg,
        "std": math.sqrt(var),
        "text": _text_status(node),
        "attribs": [
            _serialize(a, node.occ)
            for a in sorted(node.attribs.values(), key=lambda x: -x.occ)
        ],
        "children": [
            _serialize(c, node.occ)
            for c in sorted(node.children.values(), key=lambda x: -x.occ)
        ],
    }


def analyze(root_dir, bases, split_level=0, strip_namespaces=True):
    out = {}
    for base in bases:
        paths = glob(f"{root_dir}/{base}/**/*.xml", recursive=True)
        forest, ndocs = aggregate_base(paths, split_level, strip_namespaces)
        out[base] = {
            "ndocs": ndocs,
            "trees": [
                {"key": list(k), **_serialize(n, ndocs)}
                for k, n in forest.items()
            ],
        }
    return out

In [4]:
def _cardinality(node, n_parent):
    """Renvoie un symbole de cardinalité type-schéma à partir des stats."""
    optional = node["min"] == 0           # absent dans au moins une instance de parent
    multiple = node["max"] > 1
    if optional and multiple:
        return "*"   # 0..n
    if optional:
        return "?"   # 0..1
    if multiple:
        return "+"   # 1..n
    return ""        # 1..1 (obligatoire, singleton) -> pas de marqueur


def _format_line(node, prefix, is_last):
    branch = "└─ " if is_last else "├─ "
    card = _cardinality(node, None)
    name = node["name"]
    occ = node["occ"]

    # affichage adaptatif : stats détaillées seulement si non-trivial
    nontrivial = node["max"] > 1 or node["min"] == 0
    if nontrivial:
        stats = f"  [{node['min']}–{node['max']}] ~{node['avg']:.2f} ±{node['std']:.2f}"
    else:
        stats = ""

    flag = f" {card}" if card else ""
    return f"{prefix}{branch}{name}{flag}  ×{occ}{stats}"


def print_tree(tree, _prefix="", _is_root=True):
    """Affiche un arbre (un dict racine renvoyé par _serialize/analyze)."""
    if _is_root:
        # racine : pas de branche, juste le nom
        card = _cardinality(tree, None)
        flag = f" {card}" if card else ""
        nontrivial = tree["max"] > 1 or tree["min"] == 0
        stats = (f"  [{tree['min']}–{tree['max']}] ~{tree['avg']:.2f} ±{tree['std']:.2f}"
                 if nontrivial else "")
        print(f"{tree['name']}{flag}  ×{tree['occ']}{stats}")
        children = tree["children"]
        for i, c in enumerate(children):
            print_tree(c, "", i == len(children) - 1)
        return

    print(_format_line(tree, _prefix, _is_root))  # placeholder, voir ci-dessous


def print_base(base_result, base_name=""):
    """Affiche tous les arbres d'une base (sortie de analyze()[base])."""
    print(f"### {base_name}  ({base_result['ndocs']} documents)\n")
    for tree in base_result["trees"]:
        key = "/".join(tree["key"]) or "<base>"
        if tree["key"]:
            print(f"— arbre: {key} —")
        print_tree(tree)
        print()

In [5]:
def _stats_suffix(node):
    if node["max"] > 1 or node["min"] == 0:
        return f"  [{node['min']}–{node['max']}] ~{node['avg']:.2f} ±{node['std']:.2f}"
    return ""

def _card_flag(node):
    optional, multiple = node["min"] == 0, node["max"] > 1
    sym = "*" if (optional and multiple) else "?" if optional else "+" if multiple else ""
    return f" {sym}" if sym else ""

def _text_flag(node):
    st = node["text"]["status"]
    return "  «txt»" if st == "always" else "  «txt?»" if st == "sometimes" else ""

def print_tree(node, prefix="", is_last=True, is_root=True):
    line = f"{node['name']}{_card_flag(node)}  ×{node['occ']}{_stats_suffix(node)}{_text_flag(node)}"
    if is_root:
        print(line)
    else:
        branch = "└─ " if is_last else "├─ "
        print(f"{prefix}{branch}{line}")
        prefix += "   " if is_last else "│  "
    # attributs d'abord (préfixe @, style XPath), puis tags-enfants ;
    # chaînés pour que le tout dernier élément porte le connecteur └─
    attrs = node["attribs"]
    kids = node["children"]
    n = len(attrs) + len(kids)
    for i, a in enumerate(attrs):
        branch = "└─ " if i == n - 1 else "├─ "
        print(f"{prefix}{branch}@{a['name']}{_card_flag(a)}")
    for j, c in enumerate(kids):
        print_tree(c, prefix, len(attrs) + j == n - 1, is_root=False)

In [6]:
bases = ["CAPP", "CASS", "CONSTIT", "INCA", "JADE"]
schema = analyze("/mnt/data/Murphy/src", bases, split_level=0)

In [7]:
for base in bases:
    print_base(schema[base], base)

### CAPP  (1 documents)

— arbre: TEXTE_JURI_JUDI —
TEXTE_JURI_JUDI  ×1
├─ META  ×1
│  ├─ META_COMMUN  ×1
│  │  ├─ ID  ×1  «txt»
│  │  ├─ ANCIEN_ID  ×1
│  │  ├─ ORIGINE  ×1  «txt»
│  │  ├─ URL  ×1  «txt»
│  │  └─ NATURE  ×1  «txt»
│  └─ META_SPEC  ×1
│     ├─ META_JURI  ×1
│     │  ├─ TITRE  ×1  «txt»
│     │  ├─ DATE_DEC  ×1  «txt»
│     │  ├─ JURIDICTION  ×1  «txt»
│     │  ├─ NUMERO  ×1  «txt»
│     │  └─ SOLUTION  ×1
│     └─ META_JURI_JUDI  ×1
│        ├─ NUMEROS_AFFAIRES  ×1
│        │  └─ NUMERO_AFFAIRE  ×1  «txt»
│        ├─ PUBLI_BULL  ×1
│        │  └─ @publie
│        ├─ FORMATION  ×1  «txt»
│        ├─ FORM_DEC_ATT  ×1  «txt»
│        ├─ DATE_DEC_ATT  ×1  «txt»
│        ├─ SIEGE_APPEL  ×1  «txt»
│        ├─ JURI_PREM  ×1
│        ├─ LIEU_PREM  ×1
│        ├─ DEMANDEUR  ×1
│        ├─ DEFENDEUR  ×1
│        ├─ PRESIDENT  ×1
│        ├─ AVOCAT_GL  ×1
│        ├─ AVOCATS  ×1
│        └─ RAPPORTEUR  ×1
├─ TEXTE  ×1
│  ├─ BLOC_TEXTUEL  ×1
│  │  └─ CONTENU  ×1  «txt»
│  ├─ SOMMAI

In [8]:
# ── Schémas APLATIS : ce qui atterrit réellement en BDD ──────────────────────
#
# Reconstruit, à partir des arbres de `schema`, la liste des champs plats tels
# qu'ingérés — SANS field_mappings, noms BRUTS uniquement (cf. décision de scope).
#
# Convention d'aplatissement du pipeline (déduite de conf/base/parameters.yml) :
#   nom brut d'un champ = chemin des tags ancêtres en snake_case, joint par "_".
#   Puis field_mappings[document_type] canonicalise par regex (num, eli, titre…).
#   On s'arrête AVANT cette étape : on veut juger la convention de base elle-même.
#
# Deux cibles, car parameters.yml les configure différemment :
#   • mongodb.include_path = true   → le chemin complet fait partie de la clé.
#   • neo4j.include_path   = false  → seul le dernier segment est conservé.
# D'où deux fonctions, et deux profils de collision très différents.
#
# Un « champ » = une paire clé→valeur candidate au dict `metadata` :
#   - toute FEUILLE textuelle (node["text"]["status"] != "never"), et
#   - tout ATTRIBUT (@clé), qui est toujours une valeur.
# Les conteneurs purs (META, TEXTE…) ne produisent pas de champ.

def _snake(tag):
    # LEGIFRANCE : tags déjà en MAJUSCULES → simple minusculisation.
    # (les tags inline p/br sont laissés tels quels ; ils ne portent pas de métadonnée)
    return tag.lower()


def _flat_fields(node, ancestors, want_path):
    """Génère (clé_plate, kind, path_complet, text_status) pour node et sa descendance.

    kind ∈ {"leaf", "attr"} ; want_path=True → clé = chemin complet, False → dernier segment.
    """
    path = ancestors + [node["name"]]

    # attributs de ce nœud : chemin = <path>_<attr>
    for a in node["attribs"]:
        apath = path + ["@" + a["name"]]
        key = ("_".join(_snake(p.lstrip("@")) for p in apath)
               if want_path else a["name"].lower())
        yield key, "attr", "/".join(apath), None

    # ce nœud est-il une feuille-valeur ?
    if node["text"]["status"] != "never":
        key = "_".join(_snake(p) for p in path) if want_path else _snake(node["name"])
        yield key, "leaf", "/".join(path), node["text"]["status"]

    for c in node["children"]:
        yield from _flat_fields(c, path, want_path)


def _collect_schema(base_result, want_path):
    """-> (rows, collisions) ; rows = liste (key, kind, path, status) ; collisions = key -> [paths]."""
    rows = []
    for tree in base_result["trees"]:
        rows.extend(_flat_fields(tree, [], want_path))
    # collisions : une même clé plate produite par >1 chemins XML distincts
    by_key = defaultdict(list)
    for key, kind, path, _ in rows:
        by_key[key].append(path)
    collisions = {k: sorted(set(v)) for k, v in by_key.items() if len(set(v)) > 1}
    return rows, collisions


def _collision_stats(collisions):
    """Stats sur le nb de chemins par clé en collision (min/max/avg/std)."""
    counts = [len(paths) for paths in collisions.values()]   # chaque >= 2
    n = len(counts)
    if n == 0:
        return {"n_keys": 0}
    total = sum(counts)
    avg = total / n
    var = sum((c - avg) ** 2 for c in counts) / n
    return {
        "n_keys": n,
        "min": min(counts),
        "max": max(counts),
        "avg": avg,
        "std": math.sqrt(var),
        "total_paths": total,   # nb total de chemins écrasés dans des collisions
    }


def _print_schema(base_result, base_name, want_path):
    rows, collisions = _collect_schema(base_result, want_path)
    # dédup par clé, en gardant le 1er path vu (ordre document) ; tri alpha
    seen, uniq = set(), []
    for key, kind, path, status in rows:
        if key in seen:
            continue
        seen.add(key)
        uniq.append((key, kind, path, status))
    uniq.sort(key=lambda r: r[0])

    print(f"### {base_name}")
    print()
    for key, kind, path, status in uniq:
        txt = f"  «{status}»" if status and status != "always" else ""
        coll = "  ⚠" if key in collisions else ""
        print(f"  {key}{txt}{coll}")

    # synthèse collisions : compte + stats sur le nb de chemins par clé collisionnée
    st = _collision_stats(collisions)
    print()
    if st["n_keys"] == 0:
        print("  Collisions : 0 clé — aplatissement injectif ✓")
    else:
        print(f"  Collisions : {st['n_keys']} clés  "
              f"({st['total_paths']} chemins écrasés)  |  "
              f"chemins/clé  min {st['min']}  max {st['max']}  "
              f"avg {st['avg']:.2f}  std {st['std']:.2f}")
    print()


def print_schema_mongo(base_result, base_name=""):
    """Schéma plat côté MongoDB (include_path=true) : clé = chemin snake_case complet."""
    _print_schema(base_result, base_name, want_path=True)


def print_schema_neo4j(base_result, base_name=""):
    """Schéma plat côté Neo4j (include_path=false) : clé = dernier segment seul."""
    _print_schema(base_result, base_name, want_path=False)


In [9]:
# Schéma MongoDB (chemin complet) — collisions rares, seulement attribut vs sous-tag homonyme
for base in bases:
    print_schema_mongo(schema[base], base)


### CAPP

  texte_juri_judi_meta_meta_commun_id
  texte_juri_judi_meta_meta_commun_nature
  texte_juri_judi_meta_meta_commun_origine
  texte_juri_judi_meta_meta_commun_url
  texte_juri_judi_meta_meta_spec_meta_juri_date_dec
  texte_juri_judi_meta_meta_spec_meta_juri_judi_date_dec_att
  texte_juri_judi_meta_meta_spec_meta_juri_judi_form_dec_att
  texte_juri_judi_meta_meta_spec_meta_juri_judi_formation
  texte_juri_judi_meta_meta_spec_meta_juri_judi_numeros_affaires_numero_affaire
  texte_juri_judi_meta_meta_spec_meta_juri_judi_publi_bull_publie
  texte_juri_judi_meta_meta_spec_meta_juri_judi_siege_appel
  texte_juri_judi_meta_meta_spec_meta_juri_juridiction
  texte_juri_judi_meta_meta_spec_meta_juri_numero
  texte_juri_judi_meta_meta_spec_meta_juri_titre
  texte_juri_judi_texte_bloc_textuel_contenu

  Collisions : 0 clé — aplatissement injectif ✓

### CASS

  texte_juri_judi_liens_lien
  texte_juri_judi_liens_lien_cidtexte
  texte_juri_judi_liens_lien_datesignatexte
  texte_juri_judi_li

In [10]:
# Schéma Neo4j (dernier segment seul, include_path=false) — collisions massives attendues
for base in bases:
    print_schema_neo4j(schema[base], base)


### CAPP

  contenu
  date_dec
  date_dec_att
  form_dec_att
  formation
  id
  juridiction
  nature
  numero
  numero_affaire
  origine
  publie
  siege_appel
  titre
  url

  Collisions : 0 clé — aplatissement injectif ✓

### CASS

  ana  «sometimes»
  avocats  «sometimes»
  cidtexte
  contenu  ⚠
  date_dec
  date_dec_att
  datesignatexte
  ecli
  form_dec_att
  formation
  id  ⚠
  juridiction
  lien
  nature
  naturetexte
  nortexte
  num
  numero
  numero_affaire
  numtexte
  origine
  president  «sometimes»
  publie
  sct  «sometimes»
  sens
  solution
  titre
  type
  typelien
  url

  Collisions : 2 clés  (6 chemins écrasés)  |  chemins/clé  min 2  max 4  avg 3.00  std 1.00

### CONSTIT

  contenu
  date
  date_dec
  ecli
  id
  juridiction
  nature
  nature_qualifiee
  nor  ⚠
  num
  numero
  origine
  solution
  titre
  titre_jo
  url
  url_cc

  Collisions : 1 clés  (2 chemins écrasés)  |  chemins/clé  min 2  max 2  avg 2.00  std 0.00

### INCA

  ancien_id
  contenu
  date_d